# End-to-End Vulnerability Analysis on MDOF System using Incremental Dynamic Analysis

## Introduction

This Jupyter Notebook provides a structured workflow for performing Incremental Dynamic Analysis (IDA) on multi-degree-of-freedom (MDOF) structural models. Unlike static or traditional dynamic methods, IDA involves subjecting a structural model to one or more ground motion records, each scaled to multiple levels of intensity. This produces a curve of structural response (e.g., maximum interstorey drift) versus intensity level (e.g., spectral acceleration), providing a comprehensive view of the building's performance from elasticity through to global dynamic instability or collapse.

The main goals of this notebook:

1. **Compile and construct MDOF Models in OpenSees**: Define and assemble MDOF models by specifying essential structural properties, including mass, heights and nonlinear response characteristics at each degree of freedom

2. **Perform Incremental Dynamic Analysis (IDA)**: Systematically scale ground motion records using a "Hunt and Fill" algorithm. This approach rapidly identifies the collapse capacity of the system and then refines the intensity steps to capture the nonlinear transition points and the "flatline" region where the structure reaches dynamic instability.

3. **Fragility Analysis**: Postprocess the IDA curves to determine the capacity of the structure at various limit states. By analyzing the distribution of intensities that trigger specific drift thresholds, we construct fragility functions that describe the probability of exceeding a damage state as a function of ground-shaking intensity.

4. **Vulnerability Analysis**: Integrate these fragility functions with consequence models (i.e., damage-to-loss models) to determine the continuous relationship between a decision variable (such as repair cost ratio) and increasing levels of ground-shaking intensities.

The notebook provides a step-by-step guide, covering each phase from MDOF model creation and scaling-protocol configuration to analysis execution and detailed results extraction. Users should have some familiarity with Python scripting, structural dynamics, computational modeling, and performance-based earthquake engineering (PBEE) to fully benefit from this material.

---

## References

[1] Vamvatsikos, D. and Cornell, C.A. (2002), Incremental dynamic analysis. Earthquake Engng. Struct. Dyn., 31: 491-514. https://doi.org/10.1002/eqe.141

[2] Vamvatsikos D, Cornell CA. Applied Incremental Dynamic Analysis. Earthquake Spectra. 2004;20(2):523-553. doi:10.1193/1.1737737

## Initialize Libraries ##

In [ ]:
import os
import sys
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the classes necessary for structural analysis
from openquake.vmtk.units         import units              # oq-vtmk units class
from openquake.vmtk.im_calculator import IMCalculator       # oq-vmtk intensity measure processor class
from openquake.vmtk.modeller      import modeller           # oq-vmtk numerical modelling class
from openquake.vmtk.postprocessor import postprocessor      # oq-vtmk postprocessing class
from openquake.vmtk.plotter       import plotter            # oq-vmtk plotting class
from openquake.vmtk.utilities     import sorted_alphanumeric, import_from_pkl, export_to_pkl # oq-vmtk utility class

## Define Plotting Constants for OQ-VMTK Non-Native Plots ##

In [ ]:
FONTSIZE_1 = 16
FONTSIZE_2 = 14
FONTSIZE_3 = 12

LINEWIDTH_1= 3
LINEWIDTH_2= 2
LINEWIDTH_3 = 1

RESOLUTION = 500

MARKER_SIZE_1 = 100
MARKER_SIZE_2 = 60
MARKER_SIZE_3 = 10

COLOR = "#399283"

## Define Directories ##

In [ ]:
# Define the directory of the ground-motion records
gm_directory  = './records'            

# Define the main output directory
ida_directory = './output'  
os.makedirs(ida_directory, exist_ok=True)

## Load FEMA P-695 Acceleration Time-Histories and Process Intensity Measures ## 

In [ ]:
# The set of ground-motion records utilized correspond to the acceleration time-histories of FEMA P695 Far Field Set

# Input the intensity measure types required for processing
IMT = ['PGA','PGV','PGD','SA(0.3s)','SA(0.6s)','SA(1.0s)','AvgSA','AvgSA(0.3s)','AvgSA(0.6s)','AvgSA(1.0s)','AI','D595']

# Fetch the acceleration files
gmrs = sorted_alphanumeric(os.listdir(os.path.join(gm_directory, 'acc')))

# Load the time-step file
dts = pd.read_csv(os.path.join(gm_directory, 'FEMA_P695_unscaled_dts.txt'),sep=" ", header=None).values

# Load the duration file
durs = pd.read_csv(os.path.join(gm_directory, 'FEMA_P695_unscaled_durs.txt'),sep=" ", header=None).values

# Initialise the storage dictionary and name the "keys" in accordance with the defined IMT
imls = {}
for i, current_imt in enumerate(IMT):    
    imls[current_imt] = []

# Visualise the dictionary
print(imls)

# Loop over the files
for i in range(len(gmrs)):
    
    ### Load the acceleration time-histories and the corresponding time-step and durations
    current_acc = pd.read_csv(os.path.join(gm_directory,'acc',f'gm_{i+1}.txt'),sep=" ", header=None).to_numpy().flatten()
    dt_gm       = float(dts[i].item())
    dur_gm      = float(durs[i].item())

    ### Create the pseudo time-step array
    # Compute number of points safely
    num_points = int(dur_gm / dt_gm)
    
    # Generate pseudo time steps and save them unto "dts" folder
    current_dts = np.linspace(dt_gm, dur_gm, num_points) 
    os.makedirs(os.path.join(gm_directory, 'dts'),exist_ok=True)
    save_path = os.path.join(gm_directory, 'dts', f"dt_{i+1}.txt")
    np.savetxt(save_path, current_dts)
    
    ### Plot the time history only for the first ground-motion record
    if i==0:
        plt.plot(current_dts, current_acc, color = COLOR, lw=LINEWIDTH_3)
        plt.xlabel('Pseudo-time [s]', fontsize= FONTSIZE_1)
        plt.ylabel('Acceleration [g]', fontsize = FONTSIZE_1)
        plt.grid(visible=True, which='major')
        plt.grid(visible=True, which='minor')
        plt.xlim([0.00, np.max(current_dts)])
        plt.show()

    ### Initialise the Intensity Measure Calculator class
    # The class "IMCalculator" is initialised using the acceleration array and the time-step scalar value
    im_calculator = IMCalculator(current_acc,dt_gm) 

    ### Calculate amplitude-based intensity measures: peak ground acceleration, peak ground velocity, peak ground displacement 
    pga, pgv, pgd = im_calculator.get_amplitude_ims()
    
    ### Calculate duration-based intensity measures: arias intensity, cumulative absolute velocity, significant duration d595
    ai, cav, t595 = im_calculator.get_duration_ims()
     
    ### Calculate and plot the acceleration, velocity and displacement spectra 
    periods, sd, sv, sa = im_calculator.get_spectrum()

    ### Plot the spectra only for the first ground-motion record
    if i==0:

        # Acceleration Spectrum
        plt.plot(periods, sa, color = COLOR, lw=LINEWIDTH_2)
        plt.xlabel('Period [s]', fontsize= FONTSIZE_1)
        plt.ylabel('Pseudo Spectral Acceleration [g]', fontsize = FONTSIZE_1)
        plt.grid(visible=True, which='major')
        plt.grid(visible=True, which='minor')
        plt.xlim([0.00, np.max(periods)])
        plt.show()

        # Velocity Spectrum 
        plt.plot(periods, sv, color = COLOR, lw=LINEWIDTH_2)
        plt.xlabel('Period [s]', fontsize= FONTSIZE_1)
        plt.ylabel('Pseudo Spectral Velocity [m/s]', fontsize = FONTSIZE_1)
        plt.grid(visible=True, which='major')
        plt.grid(visible=True, which='minor')
        plt.xlim([0.00, np.max(periods)])
        plt.show()

        # Displacement Spectrum 
        plt.plot(periods, sd, color = COLOR, lw=LINEWIDTH_2)
        plt.xlabel('Period [s]', fontsize= FONTSIZE_1)
        plt.ylabel('Pseudo Spectral Displacement [m]', fontsize = FONTSIZE_1)
        plt.grid(visible=True, which='major')
        plt.grid(visible=True, which='minor')
        plt.xlim([0.00, np.max(periods)])
        plt.show()

    ### Calculate the spectral acceleration values at distinct periods (e.g., 0.3, 0.6 and 1.0s)
    sa03 = im_calculator.get_sa(0.3)
    sa06 = im_calculator.get_sa(0.6)
    sa10 = im_calculator.get_sa(1.0)

    ### Calculate the average spectral acceleration values at distinct periods (e.g., 0.3, 0.6 and 1.0s) 
    ### with an auto-defined range of 0.2T and 1.5T
    avgsa03 = im_calculator.get_saavg(0.3)
    avgsa06 = im_calculator.get_saavg(0.6)
    avgsa10 = im_calculator.get_saavg(1.0)

    ### Calculate the average spectral acceleration values at distinct periods (e.g., 0.3, 0.6 and 1.0s)
    ### with a user defined range 
    periods_list = np.linspace(0.1,1.0,10)
    avgsa = im_calculator.get_saavg_user_defined(periods_list)

    ### Print outputs only for first record
    if i==0:
        print(f"Peak Ground Acceleration (PGA): {pga:.4f} g")
        print(f"Peak Ground Velocity (PGV): {pgv:.4f} m/s")
        print(f"Peak Ground Displacement (PGD): {pgd:.4f} m")
    
        print(f"Arias Intensity (AI): {ai:.4f} m/s")
        print(f"Cumulative Absolute Velocity (CAV): {cav:.4f} m/s")
        print(f"Significant Duration (5%-95% AI): {t595:.4f} s")
       
        print(f"Spectral Acceleration (SA at 0.3s): {sa03:.4f} g")
        print(f"Spectral Acceleration (SA at 0.6s): {sa06:.4f} g")
        print(f"Spectral Acceleration (SA at 1.0s): {sa10:.4f} g")
    
        print(f"Average Spectral Acceleration (AvgSA at 0.3s): {avgsa03:.4f} g")
        print(f"Average Spectral Acceleration (AvgSA at 0.6s): {avgsa06:.4f} g")
        print(f"Average Spectral Acceleration (AvgSA at 1.0s): {avgsa10:.4f} g")
        print(f"Average Spectral Acceleration (AvgSA at [0.01-1.0]): {avgsa:.4f} g")

    ### Store the intensity measures in dictionary
    for j, current_imt in enumerate(IMT):

        if current_imt == 'PGA':
            imls[current_imt].append(pga) 
        elif current_imt == 'PGV':
            imls[current_imt].append(pgv) 
        elif current_imt == 'PGD':
            imls[current_imt].append(pgd)         
        elif current_imt == 'SA(0.3s)':
            imls[current_imt].append(sa03) 
        elif current_imt == 'SA(0.6s)':
            imls[current_imt].append(sa06) 
        elif current_imt == 'SA(1.0s)':
            imls[current_imt].append(sa10) 
        elif current_imt == 'AvgSA':
            imls[current_imt].append(avgsa)                                               
        elif current_imt == 'AvgSA(0.3s)':
            imls[current_imt].append(avgsa03) 
        elif current_imt == 'AvgSA(0.6s)':
            imls[current_imt].append(avgsa06) 
        elif current_imt == 'AvgSA(1.0s)':
            imls[current_imt].append(avgsa10)
        elif current_imt == 'AI':
            imls[current_imt].append(ai) 
        elif current_imt == 'AI':
            imls[current_imt].append(cav) 
        elif current_imt == 'D595':
            imls[current_imt].append(t595) 

# Export to pickle format 
export_to_pkl(os.path.join(gm_directory,'imls_femap695.pkl'), imls)

## Required MDOF Modelling Input Parameters ##

In [ ]:
# Number of storeys
number_storeys = 3 

# Relative floor heights list
floor_heights = [2.80, 3.00, 3.00]

# Relative floor masses list
floor_masses = [0.75, 0.75, 0.75] # Unit mass for SDOFs

# MDOF capacity (First row are Spectral Displacement [m] values - Second row are Spectral Acceleration [g] values)
# 3 storeys × 4 points each
storey_disps = np.array([
    [0.008, 0.015, 0.025, 0.050],   # storey 1: Brittle failure at ~1.5% drift
    [0.010, 0.018, 0.028, 0.055],   # storey 2
    [0.012, 0.022, 0.032, 0.060]    # storey 3
]) * units.m

storey_forces = np.array([
    [3.5, 5.0, 1.5, 1.5],  # storey 1: Sharp drop from 10kN to 3kN
    [4.0, 5.5, 1.7, 1.7],  # storey 2
    [4.5, 6.5, 2.0, 2.0]   # storey 3
]) * units.kN

# Flag to activate default stiffness-strength degradation and pinching4
mdof_degradation = True

# Plot the capacities to visualise the outcome of the calibration
for i in range(storey_disps.shape[0]):
   plt.plot(np.concatenate(([0.0], storey_disps[i,:])), np.concatenate(([0.0], storey_forces[i,:])), lw=LINEWIDTH_2, label = f'Storey #{i+1}')
plt.xlabel('Storey Deformation [m]', fontsize= FONTSIZE_1)
plt.ylabel('Storey Shear [kN]', fontsize = FONTSIZE_1)
plt.legend(loc = 'lower right')
plt.grid(visible=True, which='major')
plt.grid(visible=True, which='minor')
plt.xlim([0.00, 0.10])
plt.show()

### Fragility and Vulnerability Input Parameters ###

In [ ]:
# Damage thresholds (maximum peak storey drift values in rad)
damage_thresholds    =  [0.0015, 0.0030, 0.0045, 0.0135]  # Note: The damage thresholds are arbitrary and are not associated with any limit state analysis

# Define consequence model to relate structural damage to a decision variable (i.e., expected loss ratio) 
consequence_model = [0.05, 0.10, 0.60, 1.00] # damage-to-loss ratios

## Incremental Dynamic Analysis ##

Incremental Dynamic Analysis (IDA) is a computational method used in structural engineering to perform a comprehensive seismic performance evaluation. Unlike Cloud Analysis, which uses unscaled records, IDA involves subjecting a structural model to a suite of ground motion records, each systematically scaled to increasing levels of intensity (e.g., ). This process continues until the structure reaches a specified limit state or global dynamic instability, often referred to as "collapse."

The primary advantage of IDA is that it provides a continuous picture of structural behavior, from the initial elastic response through the onset of yielding, and finally to the "flatline" where a small increase in seismic intensity results in an infinite increase in structural demand. By aggregating these curves, engineers can determine the median capacity and the record-to-record variability, allowing for a highly accurate derivation of fragility curves for multiple damage states.

## Setting Up, Running IDAs and Exporting Analysis ##

In [ ]:
# Initialise MDOF storage lists
# Note: Each element in these lists will now contain the IDA curve data for that record
conv_index_list = []                
peak_disp_list  = []                
peak_drift_list = []                
peak_accel_list = []                
max_peak_drift_list = []            
max_peak_drift_dir_list = []        
max_peak_drift_loc_list = []        
max_peak_accel_list = []            
max_peak_accel_dir_list = []        
max_peak_accel_loc_list = []        

# Loop over ground-motion records, compile MDOF model and run NLTHA
gmrs = sorted_alphanumeric(os.listdir(os.path.join(gm_directory,'acc')))                         # Sort the ground-motion records alphanumerically
dts  = sorted_alphanumeric(os.listdir(os.path.join(gm_directory,'dts')))                         # Sort the ground-motion time-step files alphanumerically

# Initialize the SF Matrix: Rows = Records, Cols = Run Number
max_runs_per_record = 30
sf_matrix = np.full((len(gmrs), max_runs_per_record), np.nan)

# Loop over ground-motion records
for i in range(len(gmrs)):
    print('================================================================')
    print('============== IDA Analysing: {:d} out of {:d} =================='.format(i+1, len(gmrs)))
    print('================================================================')

    ### Compile the MDOF model (Initial build to get modal properties)
    model = modeller(number_storeys, floor_heights, floor_masses,
                     storey_disps, storey_forces*units.g, mdof_degradation)
    model.compile_model()
    
    if i == 0:
        model.plot_model()        
    
    model.do_gravity_analysis()
    num_modes = 1 if number_storeys == 1 else 3
    T, phi = model.do_modal_analysis(num_modes=num_modes, plot_modes=False)

    ### Define ground motion objects
    fnames = [os.path.join(gm_directory, 'acc', f'{gmrs[i]}')]
    fdts   = os.path.join(gm_directory, 'dts', f'{dts[i]}')

    dt_df    = pd.read_csv(fdts, sep=" ", header=None)
    dt_gm    = dt_df.iloc[1,0] - dt_df.iloc[0,0]
    t_max    = dt_df.iloc[-1,0]
    dt_ansys = dt_gm # Set the analysis time-step equal to the ground-motion time-step

    ### Run Incremental Dynamic Analysis
    # This calls the method we wrote previously which uses Hunt-Trace-Fill
    # record_ida_results is a dict: {sf: {all_outputs}}
    record_ida_results, ordered_sfs = model.do_incremental_dynamic_analysis(fnames, 
                                                                            dt_gm, 
                                                                            t_max, 
                                                                            dt_ansys,                                                                   
                                                                            target_drift=0.0135*1.5,
                                                                            initial_sf=0.1,
                                                                            hunt_step=2.0,
                                                                            max_fill_gap=0.2,
                                                                            max_runs = max_runs_per_record,
                                                                            xi=0.05)

    # Fill the matrix for the current record (row i)
    # We fill only up to the number of runs actually performed
    sf_matrix[i, :len(ordered_sfs)] = ordered_sfs
    
    ### Store the analysis results
    # We extract the components for each SF run for this record
    sfs_sorted = sorted(record_ida_results.keys())
    
    # We append the whole IDA "curve" data for this record into your storage lists
    conv_index_list.append({sf: record_ida_results[sf]['conv_index'] for sf in sfs_sorted})
    peak_drift_list.append({sf: record_ida_results[sf]['peak_drift'] for sf in sfs_sorted})
    peak_accel_list.append({sf: record_ida_results[sf]['peak_accel'] for sf in sfs_sorted})
    peak_disp_list.append({sf: record_ida_results[sf]['peak_disp'] for sf in sfs_sorted})
    
    max_peak_drift_list.append({sf: record_ida_results[sf]['max_peak_drift'] for sf in sfs_sorted})
    max_peak_drift_dir_list.append({sf: record_ida_results[sf]['max_peak_drift_dir'] for sf in sfs_sorted})
    max_peak_drift_loc_list.append({sf: record_ida_results[sf]['max_peak_drift_loc'] for sf in sfs_sorted})
    
    max_peak_accel_list.append({sf: record_ida_results[sf]['max_peak_accel'] for sf in sfs_sorted})
    max_peak_accel_dir_list.append({sf: record_ida_results[sf]['max_peak_accel_dir'] for sf in sfs_sorted})
    max_peak_accel_loc_list.append({sf: record_ida_results[sf]['max_peak_accel_loc'] for sf in sfs_sorted})

# Final storage and export (same as your original logic)
ansys_dict = {}
labels = ['T','conv_index_list', 'peak_drift_list','peak_accel_list',
          'max_peak_drift_list', 'max_peak_drift_dir_list', 
          'max_peak_drift_loc_list','max_peak_accel_list',
          'max_peak_accel_dir_list','max_peak_accel_loc_list',
          'peak_disp_list']

for label in labels:
    if label == 'T':
        ansys_dict[label] = T
    else:
        ansys_dict[label] = vars()[label]

# Save the scaling factor matrix to your results dictionary
ansys_dict['sf_matrix'] = sf_matrix

# Export the IDA results to a pickle file
export_to_pkl(os.path.join(ida_directory, 'ida_ansys_out.pkl'), ansys_dict)

print('ANALYSIS COMPLETED!')

## Calculate Scaled Ground Motion Records Intensity Measure ##

In [ ]:
# Assuming 'imls' contains the unscaled values from your loop 
# and 'sf_matrix' is the (n_gmrs x max_runs_per_record) matrix from your IDA

n_gmrs = len(gmrs)

# Storage for the expanded IMs
expanded_imls = {}

# Loop over intensity measure types
for current_imt in IMT:
    
    # Initialize a matrix of NaNs for each IMT
    # Size: (Number of Records) x (Number of Runs)
    expanded_imls[current_imt] = np.full((n_gmrs, max_runs_per_record), np.nan)

    # Loop over ground motion records
    for i in range(n_gmrs):

        # Fetch the unscaled IM value
        unscaled_val = imls[current_imt][i]

        # Fetch the row of scaling factors determined by IDA (1xn_runs) and associated with the current record
        sfs = sf_matrix[i, :] # The row of scaling factors for this GM
        
        # Apply scaling logic based on the nature of the IM
        # Linear scaling: IM_scaled = SF * IM_unscaled for amplitude-based IMs
        if current_imt in ['PGA', 'PGV', 'PGD', 'SA(0.3s)', 'SA(0.6s)', 'SA(1.0s)','AvgSA', 'AvgSA(0.3s)', 'AvgSA(0.6s)', 'AvgSA(1.0s)']:
            expanded_imls[current_imt][i, :] = sfs * unscaled_val

        # Quadratic scaling: AI_scaled = SF^2 * AI_unscaled for Arias Intensity
        elif current_imt == 'AI':
            expanded_imls[current_imt][i, :] = (sfs**2) * unscaled_val

        # Duration is scale-invariant: IM_scaled = IM_unscaled
        # We only fill columns where a run was actually performed (sf is not NaN)
        elif current_imt == 'D595':
            mask = ~np.isnan(sfs)
            expanded_imls[current_imt][i, mask] = unscaled_val

# Export the expanded dictionary
export_to_pkl(os.path.join(gm_directory, 'imls_femap695_sf.pkl'), expanded_imls)

## Post-Process the IDA Results ##

In [ ]:
# Initialise the postprocessor class
pp = postprocessor()

# Get the intensity measure matrix corresponding to an IMT key from dictionary processed in previous cell
im_key = 'AvgSA'
im_matrix = expanded_imls[im_key]

ida_dict = pp.do_incremental_dynamic_analysis(ansys_dict,
                                              im_matrix,
                                              damage_thresholds,
                                              edp_key='max_peak_drift_list',
                                              sigma_build2build=0.3,
                                              intensities=np.round(np.geomspace(0.05, 10.0, 50), 3),
                                              edp_range = np.linspace(0.00, 0.05, 101),
                                              fragility_rotation=False,
                                              rotation_percentile=0.10)

## Plot the IDA Curves ##

In [ ]:
# Initialise the plotter class
pl=plotter()

# Plot the IDA curves
pl.plot_ida_analysis(ida_dict,
                     imt_label   = 'Average Spectral Acceleration, AvgSA [g]',
                     edp_label   = r'Maximum Peak Storey Drift, $\theta_{max}$ [%]',
                     title       = 'Incremental Dynamic Analysis',
                     pFlag       = True,
                     export_path = 'figures/ida_curves.png')

## Plot the Fragility Functions ##

In [ ]:
# Plot the fragility functions
pl.plot_fragility_from_ida(ida_dict,
                           imt_label = 'Average Spectral Acceleration, AvgSA [g]',
                           title = 'Fragility Functions from IDA',
                           pFlag = True,
                           export_path = 'figures/fragility_curves.png')

## Post-Process Vulnerability Functions based on Fragility Functions ###

To derive the vulnerability, the consequence model needs to convolved with the fragility functions.  To do so, we can use the "get_vulnerability_function" method from the "postprocessor" class. Setting the uncertainty to True will additionally calculate the coefficient of variation to explicitly consider the uncertainty in the Loss|IM as per Silva et al. (2019)

In [ ]:
# In this example, the intensity measure type is 'AvgSA', to derive the vulnerability function we will use 
# the fragility function variables stored inside the "ida_dict" dictionary. The end result is a vulnerability 
# function representing the continuous relationship of the expected  structural loss ratio conditioned on 
# increasing levels of ground-shaking expressed in terms of the average spectral acceleration (in g)
structural_vulnerability = pp.get_vulnerability_function(ida_dict['fragility']['poes'],
                                                         consequence_model,
                                                         uncertainty=True)

# The output is a DataFrame with three keys: IMLs (i.e., intensity measure levels), Loss and COV
print(structural_vulnerability)

## Plot the Vulnerability Functions with Uncertainty Visualisation ##

In [ ]:
# Plot the structural vulnerability function and visualise the Beta distribution
pl.plot_vulnerability_function(structural_vulnerability['IML'],
                               structural_vulnerability['Loss'],
                               structural_vulnerability['COV'],
                               imt_label   = 'Average Spectral Acceleration, AvgSA [g]',
                               ylabel      = 'Structural Loss Ratio',
                               title       = 'Vulnerability Function from IDA',
                               pFlag       = True,
                               export_path = 'figures/vulnerability_curve.png')